# 03 · Train LoRA on Nemotron-3-Nano-30B (Google Colab Pro)

**Open this in Colab** via File → Upload, or via the Colab badge in the repo.

Requirements:
  * Colab Pro (or Pro+) with **A100 40 GB** or **L4 24 GB** runtime.
  * Kaggle API token added to Colab Secrets (left sidebar 🗝):
    * `KAGGLE_USERNAME` = your Kaggle username (e.g. `jokerdc`)
    * `KAGGLE_KEY`      = your Kaggle API key (the new short token works too)
  * Public Kaggle dataset `<you>/wonderland-sft-v1` already exists.

What this notebook does:
  1. Install deps (PEFT, TRL, datasets, bitsandbytes, mamba-ssm, causal-conv1d).
  2. Auth to Kaggle from Secrets.
  3. Download SFT data + Nemotron-3-Nano-30B base model via kagglehub.
  4. QLoRA fine-tune (4-bit base + LoRA rank 32). Auto-picks BF16 if GPU has ≥ 70 GB.
  5. Run held-out validation on 300 rows with the same metric the grader uses.
  6. Save the adapter and pack `submission.zip` for download.

## 0. GPU check

In [ ]:
import subprocess, torch
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
print(f'torch.cuda.device_count(): {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {torch.cuda.get_device_name(i)}  '
          f'{props.total_memory / 1024**3:.1f} GiB  sm_{props.major}{props.minor}')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout)

## 1. Install dependencies

Colab has internet, so this is the standard pip install path. `mamba-ssm` and
`causal-conv1d` are compiled extensions; using the maintainer's hosted wheels
avoids the slow build-from-source path.

In [ ]:
%pip install -q -U peft accelerate trl datasets bitsandbytes polars huggingface-hub kagglehub
# Mamba kernels — pin to known-good versions matching Colab's torch 2.5/2.6 + cu12.
%pip install -q --no-build-isolation mamba-ssm==2.3.1 causal-conv1d==1.6.1 einops

## 2. Kaggle auth via Colab Secrets

Open the 🗝 panel on the left, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Secrets
with notebook access enabled, then run this cell. Falls back to a manual paste
if Secrets aren't set.

In [ ]:
import os, json, getpass
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('loaded Kaggle creds from Colab Secrets')
except Exception as e:
    print(f'Colab Secrets unavailable ({e}); falling back to manual entry')
    if 'KAGGLE_USERNAME' not in os.environ:
        os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ').strip()
    if 'KAGGLE_KEY' not in os.environ:
        os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle key: ').strip()

# Also drop a ~/.kaggle/kaggle.json so kagglehub / kaggle CLI both find it.
kdir = os.path.expanduser('~/.kaggle')
os.makedirs(kdir, exist_ok=True)
with open(f'{kdir}/kaggle.json', 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(f'{kdir}/kaggle.json', 0o600)
print('kaggle.json written for', os.environ['KAGGLE_USERNAME'])

## 3. Download SFT data + base model

In [ ]:
import kagglehub
import polars as pl

# SFT data — the dataset we prepared locally.
SFT_USER = os.environ['KAGGLE_USERNAME']
SFT_DIR = kagglehub.dataset_download(f'{SFT_USER}/wonderland-sft-v1')
print('SFT dir:', SFT_DIR)

import glob
SFT_PATH = next(iter(glob.glob(f'{SFT_DIR}/**/sft_v1.parquet', recursive=True)))
df = pl.read_parquet(SFT_PATH)
print('total rows:', df.height)
print(df.group_by(['category', 'source']).agg(pl.len().alias('n')).sort(['category', 'source']))

# Base model — large (~63 GiB); progress bar will appear.
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print('model dir:', MODEL_PATH)

## 4. Stratified 90/10 train/val split + HF Dataset

In [ ]:
import json
from datasets import Dataset

SPLIT_SEED = 0
VAL_FRACTION = 0.10

df = df.with_row_index('_row')
val_parts, train_parts = [], []
for cat in sorted(df['category'].unique().to_list()):
    sub = df.filter(pl.col('category') == cat).sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    n_val = max(1, int(round(sub.height * VAL_FRACTION)))
    val_parts.append(sub.head(n_val))
    train_parts.append(sub.tail(sub.height - n_val))
df_val = pl.concat(val_parts).sort('_row').drop('_row')
df_train = pl.concat(train_parts).sort('_row').drop('_row')
df = df.drop('_row')
print(f'train rows: {df_train.height}   val rows: {df_val.height}')

ds_train = Dataset.from_list([{'messages': json.loads(m)} for m in df_train['messages'].to_list()])
print(ds_train)

## 5. Load model — QLoRA on A100/L4, BF16 if you have ≥ 70 GiB

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
LORA_RANK = 32

vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
use_4bit = vram_gib < 70
print(f'VRAM={vram_gib:.1f} GiB → load_strategy={"4-bit QLoRA" if use_4bit else "BF16"}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

load_kwargs = dict(
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
if use_4bit:
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
else:
    load_kwargs['dtype'] = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=16,
    target_modules=r'.*\\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. SFT training

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=25,
    save_strategy='no',
    max_seq_length=2048,
    packing=False,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds_train, tokenizer=tokenizer)
trainer.train()

## 7. Held-out validation (sample of 300 rows)

Inline copy of the local metric so the notebook stays self-contained.

In [ ]:
import re

VAL_CAP = 300
ADAPTER_DIR = OUTPUT_DIR + '/adapter'
model.save_pretrained(ADAPTER_DIR)

cats = sorted(df_val['category'].unique().to_list())
per_cat = max(1, VAL_CAP // len(cats))
df_val_eval = pl.concat([
    df_val.filter(pl.col('category') == c).sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED).head(per_cat)
    for c in cats
])
print(f'scoring {df_val_eval.height} rows')

_BOXED_RE = re.compile(r'\\boxed\{(.*?)\}', re.DOTALL)
_NUM_RE = re.compile(r'-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?')
def extract_answer(text):
    boxed = _BOXED_RE.findall(text)
    if boxed: return boxed[-1].strip()
    nums = _NUM_RE.findall(text)
    if nums: return nums[-1]
    return text.strip()
def _f(s):
    try: return float(s)
    except: return None
def is_correct(pred, target):
    p = extract_answer(pred).strip(); t = target.strip()
    if p == t: return True
    pn, tn = _f(p), _f(t)
    if pn is None or tn is None: return False
    return abs(pn - tn) / max(abs(tn), 1e-12) <= 1e-2

model.eval()
scored = []
for row in df_val_eval.iter_rows(named=True):
    msgs = json.loads(row['messages'])
    inf_msgs = [m for m in msgs if m['role'] != 'assistant']
    text = tokenizer.apply_chat_template(inf_msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, do_sample=False, max_new_tokens=2048,
                              pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)
    scored.append({'category': row['category'], 'target': row['answer'],
                   'prediction': gen, 'correct': is_correct(gen, row['answer'])})

scored_df = pl.DataFrame(scored)
print(f'overall val acc: {scored_df["correct"].mean():.4f}')
print(scored_df.group_by('category').agg([
    pl.len().alias('n'), pl.col('correct').mean().alias('acc')
]).sort('category'))

## 8. Package submission.zip + download

In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR + '/submission', 'zip', ADAPTER_DIR)
print('wrote', OUTPUT_DIR + '/submission.zip',
      f'({os.path.getsize(OUTPUT_DIR + "/submission.zip")/1024/1024:.2f} MB)')

# Download to your local machine.
from google.colab import files
files.download(OUTPUT_DIR + '/submission.zip')